<a href="https://colab.research.google.com/github/PoojaKumariR-student/Supply-Chain-Demand-Forecasting/blob/main/Supply_Chain_Demand_Forecasting_Project_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 1 - Loading the libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# **Step 2 - Loading Dataset**

In [ ]:
df = pd.read_csv("/content/demand_forecasting_dataset.csv")
df

# **Step 3 - Undersatanding the Dataset**

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info

In [ ]:
df.describe()

In [ ]:
df.columns

# **Step 4 - Cleaning the Data**

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(10,5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

In [ ]:
df.drop_duplicates()

In [ ]:
df = df.drop(columns="economic_index")
df = df.drop(columns="discount_percentage")

In [ ]:
df = df.drop(columns="region_Europe")
df = df.drop(columns="region_North America")

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.describe()

# **Importing Libraries**

In [ ]:
!pip install pytorch-lightning
!pip install pytorch-forecasting
!pip install shap

In [ ]:
!pip install tensorflow

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

import shap

In [ ]:
df.dtypes

In [ ]:
le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col].astype(str))

# **Step 6 - Data Visualization**

Demand Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['future_demand'], bins=30, kde=True)

plt.title("Future Demand Distribution")
plt.show()

Sales Trend

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df['sales_units'])
plt.title("Sales Units Trend")
plt.xlabel("Records")
plt.ylabel("Sales Units")

plt.show()

Correlation Heatmap

In [ ]:
plt.figure(figsize=(14,10))

sns.heatmap(df.corr(numeric_only=True),
            cmap='coolwarm',
            annot=False)

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(y=df['sales_revenue'])
plt.title("Sales Revenue Distribution")
plt.show()

# **Step 6 - Model Training and Evaluation**

In [ ]:
X = df.drop(['future_demand','date'], axis=1)
y = df['future_demand']

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

# **Step 7 -LSTM Model**

Reshaping Data for LSTM

Training the model

In [ ]:
X_train_lstm = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test_lstm = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

In [ ]:
model = Sequential()

model.add(
    LSTM(
        32,
        return_sequences=True,
        input_shape=(30,1)
    )
)

model.add(Dropout(0.2))

model.add(LSTM(32))

model.add(Dropout(0.2))

model.add(Dense(1))

In [ ]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.summary()

In [ ]:
history = model.fit(
    X_train_lstm,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Prediction

In [ ]:
predictions = model.predict(X_test_lstm)

Evaluation

In [ ]:
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE :", rmse)
print("R2 Score :", r2)

Graph for Actual vs Predicted

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(y_test.values[:100],
         label='Actual Demand')
plt.plot(predictions[:100],
         label='Predicted Demand')

plt.legend()
plt.title("Actual vs Predicted Demand")
plt.show()

# **Step 8 - Transformer Model**

Importing Necessary Libraries

In [ ]:
from tensorflow.keras.layers import MultiHeadAttention
from tensorflow.keras.layers import LayerNormalization
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Flatten
from tensorflow.keras.models import Model

In [ ]:
inputs = Input(
    shape=(X_train_lstm.shape[1],1)
)

attention = MultiHeadAttention(
    num_heads=2,
    key_dim=2
)(inputs, inputs)

x = LayerNormalization()(attention)

x = Dense(64, activation='relu')(x)

x = Flatten()(x)

outputs = Dense(1)(x)

transformer_model = Model(inputs, outputs)

transformer_model.compile(
    optimizer='adam',
    loss='mse'
)

transformer_model.summary()

In [ ]:
transformer_model.fit(
    X_train_lstm,
    y_train,
    epochs=5,
    batch_size=32
)

# **Step 9 - SHAP**

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

# Small sample for faster execution
sample_data = X_test_lstm[:20]

# Prediction function
def predict_fn(data):

    data = data.reshape(
        data.shape[0],
        X_test_lstm.shape[1],
        1
    )

    return model.predict(data).flatten()

# Flatten input for SHAP
sample_flat = sample_data.reshape(sample_data.shape[0], -1)

background = sample_flat[:10]

# Kernel Explainer
explainer = shap.KernelExplainer(
    predict_fn,
    background
)

# Generate SHAP values
shap_values = explainer.shap_values(
    sample_flat[:20]
)

# Convert properly
shap_values = np.array(shap_values)

# Plot
plt.figure(figsize=(10,6))

shap.summary_plot(
    shap_values,
    sample_flat[:20],
    feature_names=list(X.columns)
)

plt.show()

# **Step 10 - Temporal Fusion Transformer (TFT)**

In [ ]:
from pytorch_forecasting import TemporalFusionTransformer

print("Temporal Fusion Transformer Imported")

Forecasting Visualization

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(predictions[:200],
         label='Forecasted Demand')

plt.title("Supply Chain Demand Forecasting")

plt.xlabel("Samples")

plt.ylabel("Demand")

plt.legend()

plt.show()
